# Hamiltonian Flow Matching - Fixed-Center Three-Body Co-Orbital Bridge

This notebook transports two moving bodies around a fixed central body.  The learned configuration is
`q = (q1_x, q1_y, q2_x, q2_y)`, while the third body stays pinned at the origin and only enters the
Hamiltonian.

The endpoint distributions are uniform squares around two co-orbital means.  The terminal means are a
counterclockwise rotation of the initial means by `alpha`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../../../'))
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader,Dataset
from torchdyn.core import NeuralODE

from torchcfm.hamiltonian import (
    MeanStdBVPGaussianPath, flow_matching_loss,
    FixedCenterThreeBodyPotential,
    to_numpy, as_particles as _as_particles, make_hamiltonian_node, make_mean_std_bvp_path,
    solve_bvp_paths, train_on_cached_path_pairs, train_on_ot_pairs as _train_on_ot_pairs,
    simulate_model_trajectory, cached_mean_trajectory, trajectory_hamiltonian,
)
from torchcfm.optimal_transport import OTPlanSampler,wasserstein
from torchcfm.models.models_v2 import MLP,FourierTimeResidualMLP
from torchcfm.utils import torch_wrapper

from ema_pytorch import EMA

if torch.cuda.is_available() and torch.cuda.device_count() > 2:
    device = torch.device('cuda:2')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

import copy


In [ ]:
torch.manual_seed(42)
np.random.seed(42)

# Learned state: two moving bodies in 2D.  The fixed central body is not part
# of the model state, but it appears in the Hamiltonian and plots.
n_moving = 2
particle_dim = 2
dim = n_moving * particle_dim

# Co-orbital bridge geometry.
r = 5
alpha = math.pi/2
square_width = 0.5
endpoint_distribution = "gaussian"
endpoint_std = square_width
fixed_position = torch.tensor([0.0, 0.0], device=device)

source_positions = torch.tensor(
    [[-2*r, 0.0], [-r, 0.0]],
    device=device,
    dtype=torch.get_default_dtype(),
)
rotation_matrix = torch.tensor(
    [
        [math.cos(alpha), -math.sin(alpha)],
        [math.sin(alpha), math.cos(alpha)],
    ],
    device=device,
    dtype=source_positions.dtype,
)
target_positions = source_positions @ rotation_matrix.T
source_mean = source_positions.reshape(-1)
target_mean = target_positions.reshape(-1)

# Hamiltonian constants.  `softening_epsilon` enters the softened distance
# sqrt(||x-y||^2 + epsilon^2).
G = 1.0
central_mass = 20.0
moving_mass = 1.0
softening_epsilon = 0.01

# Training controls. Increase n_dataset/n_epochs/n_iters for production runs.
batch_size = 2**9
n_dataset = 2**9
n_epochs = 40
n_iters = 248
n_warmup_iters = 5000
batch_size_warmup = 1024
lr = 5e-4
bvp_sigma = 1e-2
quadrature_order = 3
n_steps = 100
tol = 1e-2
eval_batch = 2500
diagnostic_batch = eval_batch
block_size = 2
solve_t_span = torch.linspace(0, 1, n_steps + 1, device=device)

print(f'device: {device}')
print(f'dim: {dim}')
print(f'source means: {source_positions.tolist()}')
print(f'target means: {target_positions.tolist()}')
print(f'endpoint distribution: {endpoint_distribution}, endpoint_std: {endpoint_std}')
print(f'square width: {square_width}, alpha: {alpha:.4f} rad')

## Softened Fixed-Center Three-Body Hamiltonian

For moving bodies `q1`, `q2` and fixed center `c`, the potential is

$$
V(q) = -\frac{GMm}{\sqrt{\|q_1-c\|^2+\epsilon^2}}
       -\frac{GMm}{\sqrt{\|q_2-c\|^2+\epsilon^2}}
       -\frac{Gm^2}{\sqrt{\|q_1-q_2\|^2+\epsilon^2}}.
$$

The BVP path uses `q'' = -\nabla V(q)`.

In [ ]:
batch_size

In [ ]:
potential = FixedCenterThreeBodyPotential(
    fixed_position=fixed_position,
    G=G,
    central_mass=central_mass,
    moving_mass=moving_mass,
    epsilon=softening_epsilon,
)

q_test = source_mean.reshape(1, -1)
print(f"V(source_mean) = {potential.energy(q_test).item():.6f}")
print(f"force(source_mean) = {potential.force(q_test).reshape(n_moving, particle_dim)}")

In [39]:
class TotalDataset(Dataset):
        
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return self.samples.shape[0]
    def __getitem__(self,idx):
        return self.samples[idx]

source_dist = torch.distributions.Normal(source_mean, endpoint_std)
target_dist = torch.distributions.Normal(target_mean, endpoint_std)


source_data = source_dist.sample((eval_batch,))
target_data = target_dist.sample((eval_batch,))

source_loader = DataLoader(TotalDataset(source_data),batch_size = batch_size, shuffle = True, drop_last = True)
target_loader = DataLoader(TotalDataset(target_data),batch_size = batch_size, shuffle = True, drop_last = True)

In [ ]:


def sample_square_around(means, n, width=square_width):
    means = means.to(device)
    offsets = (torch.rand((n, *means.shape), device=device, dtype=means.dtype) - 0.5) * width
    return (means.unsqueeze(0) + offsets).reshape(n, -1)


def sample_endpoint(dist, means, n):
    if endpoint_distribution == "gaussian":
        return dist.sample((n,)).to(device)
    if endpoint_distribution == "uniform_square":
        return sample_square_around(means, n)
    raise ValueError(f"Unknown endpoint_distribution: {endpoint_distribution!r}")


def sample_source(n):
    return sample_endpoint(source_dist, source_positions, n)


def sample_target(n):
    return sample_endpoint(target_dist, target_positions, n)


def as_particles(q):
    return _as_particles(q, n_moving, particle_dim)


def particle_colors():
    return [plt.cm.tab10(0), plt.cm.tab10(3)]


def draw_endpoint_regions(ax):
    if endpoint_distribution == "gaussian":
        for positions, color, label, linestyle in [
            (to_numpy(source_positions), "black", "source std", "-"),
            (to_numpy(target_positions), "0.45", "target std", "--"),
        ]:
            for idx, center in enumerate(positions):
                ax.add_patch(
                    plt.Circle(
                        (center[0], center[1]),
                        endpoint_std,
                        fill=False,
                        color=color,
                        alpha=0.75,
                        linewidth=1.4,
                        linestyle=linestyle,
                        label=label if idx == 0 else None,
                    )
                )
                ax.errorbar(
                    center[0],
                    center[1],
                    xerr=endpoint_std,
                    yerr=endpoint_std,
                    fmt="none",
                    ecolor=color,
                    elinewidth=1.1,
                    capsize=3,
                    alpha=0.75,
                )
        return

    if endpoint_distribution == "uniform_square":
        half = square_width / 2.0
        for positions, color, label in [
            (to_numpy(source_positions), "black", "source square"),
            (to_numpy(target_positions), "0.45", "target square"),
        ]:
            for idx, center in enumerate(positions):
                rect = plt.Rectangle(
                    (center[0] - half, center[1] - half),
                    square_width,
                    square_width,
                    fill=False,
                    color=color,
                    alpha=0.65,
                    linewidth=1.4,
                    label=label if idx == 0 else None,
                )
                ax.add_patch(rect)
        return

    raise ValueError(f"Unknown endpoint_distribution: {endpoint_distribution!r}")


def endpoint_plot_title():
    if endpoint_distribution == "gaussian":
        return "Gaussian endpoint samples"
    if endpoint_distribution == "uniform_square":
        return "Uniform square endpoint samples"
    return f"Endpoint samples ({endpoint_distribution})"


def plot_endpoint_samples(x0, x1, title=None, n_show=500):
    x0p = to_numpy(as_particles(x0[:n_show]))
    x1p = to_numpy(as_particles(x1[:n_show]))
    colors = particle_colors()

    fig, ax = plt.subplots(figsize=(7, 6))
    for i, color in enumerate(colors):
        ax.scatter(x0p[:, i, 0], x0p[:, i, 1], s=8, color=color, alpha=0.35, label=f"source q{i + 1}")
        ax.scatter(x1p[:, i, 0], x1p[:, i, 1], s=8, color=color, marker="x", alpha=0.55, label=f"target q{i + 1}")
    draw_endpoint_regions(ax)
    ax.scatter([fixed_position[0].item()], [fixed_position[1].item()], marker="*", s=180, color="gold", edgecolor="black", label="fixed center")
    ax.plot(to_numpy(source_positions[:, 0]), to_numpy(source_positions[:, 1]), color="black", linewidth=1.4, alpha=0.6)
    ax.plot(to_numpy(target_positions[:, 0]), to_numpy(target_positions[:, 1]), color="0.35", linewidth=1.4, alpha=0.6)
    ax.axis("equal")
    ax.grid(alpha=0.25)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title or endpoint_plot_title())
    ax.legend(markerscale=2, fontsize=8, ncols=2)
    plt.tight_layout()
    plt.show()


x0_vis = source_data
x1_vis = target_data
plot_endpoint_samples(x0_vis, x1_vis,n_show = eval_batch)

## BVP Path, Models, and Training Helpers

The nonlinear reference path is solved with `MeanStdBVPGaussianPath`.  Its cache stores successful
endpoint pairs, and training samples `(x_t, u_t)` from those cached BVP paths.

In [41]:



ot_sampler = OTPlanSampler(method='exact')

# fwd_model_warmup = MLP(dim, out_dim=dim, w=512, time_varying=True).to(device)
# bwd_model_warmup = MLP(dim, out_dim=dim, w=512, time_varying=True).to(device)
fwd_model_warmup = FourierTimeResidualMLP(dim=dim,out_dim=dim,w=512,hidden=6,m=6,time_varying=True).to(device)
bwd_model_warmup = FourierTimeResidualMLP(dim=dim,out_dim=dim,w=512,hidden=6,m=6,time_varying=True).to(device)
fwd_optimizer = torch.optim.Adam(fwd_model_warmup.parameters(), lr=1e-5)
bwd_optimizer = torch.optim.Adam(bwd_model_warmup.parameters(), lr=1e-5)

fwd_losses = []
bwd_losses = []


def make_node(model):
    return make_hamiltonian_node(model)


def make_path(mu_guess=None, mu_dot_guess=None):
    return make_mean_std_bvp_path(
        potential,
        sigma=bvp_sigma,
        n_steps=n_steps,
        tol=tol,
        quadrature_order=quadrature_order,
        mu_guess=mu_guess,
        mu_dot_guess=mu_dot_guess,
        use_monte_carlo = True,
        monte_carlo_samples = 100,
        
    )


def solve_paths(x0, x1, label="path", mu_guess=None, mu_dot_guess=None):
    def make_path_with_guesses():
        return make_path(mu_guess=mu_guess, mu_dot_guess=mu_dot_guess)

    return solve_bvp_paths(make_path_with_guesses, x0, x1, label=label, description="fixed-center three-body BVPs")


def trajectory_velocity_guesses(model, traj, t_span, x0, x1, reverse=True):
    if traj.ndim != 3:
        raise ValueError(f"traj must have shape (time, batch, dim); got {tuple(traj.shape)}.")
    if traj.shape[0] != t_span.numel():
        raise ValueError(f"traj has {traj.shape[0]} time steps, but t_span has {t_span.numel()}.")
    if traj.shape[0] != n_steps + 1:
        raise ValueError(f"traj must have n_steps + 1 = {n_steps + 1} time steps; got {traj.shape[0]}.")

    batch, guess_dim = traj.shape[1], traj.shape[2]
    x0_flat = x0.detach().reshape(x0.shape[0], -1)
    x1_flat = x1.detach().reshape(x1.shape[0], -1)
    if x0_flat.shape != (batch, guess_dim) or x1_flat.shape != (batch, guess_dim):
        raise ValueError(
            f"Endpoint shapes must flatten to {(batch, guess_dim)}; "
            f"got {tuple(x0_flat.shape)} and {tuple(x1_flat.shape)}."
        )

    was_training = model.training
    model.eval()
    with torch.no_grad():
        t_values = t_span.to(device=traj.device, dtype=traj.dtype).reshape(-1)
        velocities = []
        for k, t_value in enumerate(t_values):
            t_col = t_value.reshape(1, 1).expand(batch, 1)
            velocities.append(model(torch.cat([traj[k], t_col], dim=-1)))
        mu_dot = torch.stack(velocities, dim=0)
    if was_training:
        model.train()

    mu = traj.detach()
    if reverse:
        mu = torch.flip(mu, dims=(0,))
        mu_dot = -torch.flip(mu_dot, dims=(0,))

    mu_guess = mu.permute(1, 0, 2).contiguous().clone()
    mu_dot_guess = mu_dot.permute(1, 0, 2).contiguous().detach().clone()
    x0_flat = x0_flat.to(device=mu_guess.device, dtype=mu_guess.dtype)
    x1_flat = x1_flat.to(device=mu_guess.device, dtype=mu_guess.dtype)
    mu_guess[:, 0, :] = x0_flat
    mu_guess[:, -1, :] = x1_flat

    assert mu_guess.shape == mu_dot_guess.shape == (batch, n_steps + 1, guess_dim)
    assert torch.allclose(mu_guess[:, 0, :], x0_flat)
    assert torch.allclose(mu_guess[:, -1, :], x1_flat)
    return to_numpy(mu_guess), to_numpy(mu_dot_guess)


def train_on_cached_paths(model, optimizer, path, x0, x1, n_steps_train, label, ema=None, log_every=200):
    return train_on_cached_path_pairs(
        model, optimizer, path, x0, x1, n_steps_train, label, ema=ema,
        batch_size=batch_size,
        device=device,
        log_every=log_every,
        no_pairs_message="no successful BVP pairs to train on",
    )


def train_on_ot_pairs(model, optimizer, x0, x1, n_steps_train, label, log_every=500):
    return _train_on_ot_pairs(
        model, optimizer, x0, x1, n_steps_train, label,
        batch_size=batch_size,
        device=device,
        log_every=log_every,
    )

## Initial BVP Dataset

In [ ]:
x0_all = sample_source(batch_size)
x1_all = sample_target(batch_size)
x0_coupled, x1_coupled = ot_sampler.sample_plan(x0_all, x1_all)
x0_coupled,x1_coupled = x0_all,x1_all
path, x0_coupled, x1_coupled, states = solve_paths(x0_coupled, x1_coupled, label='initial')
# n_dataset = x0_coupled.shape[0]





In [ ]:
plot_idx = np.linspace(0, x0_coupled.shape[0] - 1, min(10, x0_coupled.shape[0])).astype(int)
t_state = to_numpy(path.t_grid)
colors = particle_colors()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for idx in plot_idx:
    q_path = to_numpy(as_particles(states[idx, :, :dim]))
    sigma = to_numpy(states[idx,:,2*dim])
    # speed = np.linalg.norm(to_numpy(states[idx, :, dim:]).reshape(n_steps + 1, n_moving, particle_dim), axis=-1)
    # axes[0].plot(q_path[:,0],q_path[:,1],linewidth = 1.8,alpha = 0.8, label= f'path {idx}')
    
    for pidx, color in enumerate(colors):
        axes[0].plot(q_path[:, pidx, 0], q_path[:, pidx, 1], color=color, alpha=0.55, linewidth=1.2)
        axes[1].plot(t_state, sigma, color=color, alpha=0.55, linewidth=1.2)
    # axes[1].plot(t_state, speed.mean(axis=-1), alpha=0.75)

axes[0].scatter([fixed_position[0].item()], [fixed_position[1].item()], marker='*', s=150, color='gold', edgecolor='black')
axes[0].axis('equal')
axes[0].grid(alpha=0.25)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Initial cached BVP trajectories')
axes[1].set_xlabel('t')
axes[1].set_ylabel('sigma(t)')
axes[1].set_title('BVP sigma profiles')
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Warm Start on Straight OT Pairs

In [126]:
# warm_x0 = sample_source(batch_size_warmup)
# warm_x1 = sample_target(batch_size_warmup)
# warm_x0, warm_x1 = ot_sampler.sample_plan(warm_x0, warm_x1)

# print('Warm-start forward model on straight OT pairs...')
# fwd_losses.extend(train_on_ot_pairs(fwd_model_warmup, fwd_optimizer, warm_x0, warm_x1, n_warmup_iters, 'fwd warm'))
# print('Warm-start backward model on straight OT pairs...')
# bwd_losses.extend(train_on_ot_pairs(bwd_model_warmup, bwd_optimizer, warm_x1, warm_x0, n_warmup_iters, 'bwd warm'))

## Alternating Forward/Backward Training

In [127]:
def plot_learned_three_body_trajectories(traj, reference, title, n_show=500, n_lines=100):
    # n_show = min(n_show, traj.shape[1], reference.shape[0])
    n_show = traj.shape[1]
    # n_lines = min(n_lines, traj.shape[1])
    n_lines = traj.shape[1]
    traj_particles = to_numpy(as_particles(traj.detach().cpu()))
    ref_particles = to_numpy(as_particles(reference.detach().cpu()))
    colors = particle_colors()

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

    for pidx, color in enumerate(colors):
        axes[0].scatter(traj_particles[0, :n_show, pidx, 0], traj_particles[0, :n_show, pidx, 1], s=8, color=color, alpha=0.35, label=f'q{pidx + 1} start')
        axes[0].scatter(ref_particles[:n_show, pidx, 0], ref_particles[:n_show, pidx, 1], s=7, color=color, alpha=0.18)
        axes[0].scatter(traj_particles[-1, :n_show, pidx, 0], traj_particles[-1, :n_show, pidx, 1], s=10, color='black', marker='x', alpha=0.55)
    # draw_endpoint_squares(axes[0])
    axes[0].set_title(f'{title}: terminal samples vs reference')

    for i in range(n_lines):
        for pidx, color in enumerate(colors):
            axes[1].plot(traj_particles[:, i, pidx, 0], traj_particles[:, i, pidx, 1], color=color, alpha=0.05, linewidth=0.2)
        # axes[1].plot(traj_particles[:, i, :, 0].T, traj_particles[:, i, :, 1].T, color='0.35', alpha=0.025, linewidth=0.5)
    axes[1].set_title(f'{title}: individual learned trajectories')

    for ax in axes:
        ax.scatter([fixed_position[0].item()], [fixed_position[1].item()], marker='*', s=160, color='gold', edgecolor='black', label='fixed center')
        ax.axis('equal')
        ax.grid(alpha=0.25)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.legend(markerscale=2, fontsize=8)

    plt.tight_layout()
    plt.show()


In [128]:
ema_beta = 0.9999
ema_update_after_step = 0
ema_update_every = 1
ema_freeze_batches = 0
ema_diag_every = ema_update_every

# fwd_optimizer = torch.optim.AdamW(fwd_model.parameters(), lr=1e-5)
# bwd_optimizer = torch.optim.AdamW(bwd_model.parameters(), lr=1e-5)


def mean_param_delta(model, ema_model):
    deltas = []
    with torch.no_grad():
        for param, ema_param in zip(model.parameters(), ema_model.parameters()):
            deltas.append((param - ema_param).abs().mean())
    if not deltas:
        return 0.0
    return torch.stack(deltas).mean().item()


def print_ema_diagnostic(context, generation_model=None):
    fwd_step = int(fwd_ema.step.item()) if fwd_ema is not None else -1
    bwd_step = int(bwd_ema.step.item()) if bwd_ema is not None else -1
    fwd_delta = mean_param_delta(fwd_model, fwd_ema.ema_model) if fwd_ema is not None else float('nan')
    bwd_delta = mean_param_delta(bwd_model, bwd_ema.ema_model) if bwd_ema is not None else float('nan')
    generation = f', generation={generation_model}' if generation_model is not None else ''
    print(
        f'{context}: fwd_ema_step={fwd_step}, bwd_ema_step={bwd_step}, '
        f'fwd_delta={fwd_delta:.3e}, bwd_delta={bwd_delta:.3e}{generation}'
    )

fwd_model = FourierTimeResidualMLP(dim=dim,out_dim=dim,w=512,hidden=6,m=6,time_varying=True).to(device)
bwd_model = FourierTimeResidualMLP(dim=dim,out_dim=dim,w=512,hidden=6,m=6,time_varying=True).to(device)
fwd_losses = []
bwd_losses = []
# fwd_model = copy.deepcopy(fwd_model_warmup)
# bwd_model = copy.deepcopy(bwd_model_warmup)
# fwd_ema = None
# bwd_ema = None
fwd_ema = EMA(
    fwd_model,
    beta=ema_beta,
    update_after_step=ema_update_after_step,
    update_every=ema_update_every,
    min_value=ema_beta,
).to(device)
# bwd_ema = None
bwd_ema = EMA(
    bwd_model,
    beta=ema_beta,
    update_after_step=ema_update_after_step,
    update_every=ema_update_every,
    min_value=ema_beta,
).to(device)

# # Initialize EMA from the warm-started online models once, then let the slow schedule control drift.
# fwd_ema.update()
# bwd_ema.update()
# print_ema_diagnostic('EMA initialized from warm-start models')


In [ ]:
last_fwd_path = None
last_bwd_path = None
last_fwd_states = None
last_bwd_states = None
last_fwd_x0, last_fwd_x1 = None, None
last_bwd_x0, last_bwd_x1 = None, None
eval_t_span = torch.linspace(0, 1, 100, device=device)

x0_eval = sample_source(eval_batch//10)
x1_eval = sample_target(eval_batch//10)
bwd_optimizer = torch.optim.AdamW(bwd_model.parameters(), lr=1e-4)
fwd_optimizer = torch.optim.AdamW(fwd_model.parameters(), lr=1e-4)


def select_generation_model(ema, online_model, ema_name, online_name):
    if ema is not None:
        generation_model = ema.ema_model
        generation_label = ema_name
    else:
        generation_model = online_model
        generation_label = online_name
    generation_model.eval()
    return generation_model, generation_label


def generate_fwd_path_cache(epoch, batch_idx, y_target, x_source, generation_model):
    mu_guess, mu_dot_guess = None, None
    if epoch == 0:
        x_source, y_target = ot_sampler.sample_plan(x_source, y_target)
        pass
    else:
        generation_node = make_node(generation_model)
        with torch.no_grad():
            bwd_traj = generation_node.trajectory(y_target, t_span=solve_t_span)
        generated_source = bwd_traj[-1].detach()
        generated_source, x_source, bwd_traj_guess, _ = ot_sampler.sample_plan_with_labels(
            generated_source,
            x_source,
            y0=bwd_traj.permute(1, 0, 2).contiguous(),
            y1=None,
        )
        y_target = bwd_traj_guess[:, 0, :].detach()
        bwd_traj_guess = bwd_traj_guess.permute(1, 0, 2).contiguous()
        mu_guess, mu_dot_guess = trajectory_velocity_guesses(
            generation_model,
            bwd_traj_guess,
            solve_t_span,
            x_source,
            y_target,
            reverse=True,
        )

    path, x0_keep, x1_keep, states = solve_paths(
        x_source,
        y_target,
        label=f'epoch {epoch} batch {batch_idx} fwd',
        mu_guess=mu_guess,
        mu_dot_guess=mu_dot_guess,
    )
    return {
        'path': path,
        'x0': x0_keep,
        'x1': x1_keep,
        'states': states,
        'label': f'epoch {epoch} batch {batch_idx} fwd',
    }


def generate_bwd_path_cache(epoch, batch_idx, x_source, y_target, generation_model):
    mu_guess, mu_dot_guess = None, None
    if epoch == 0:
        x_source, y_target = ot_sampler.sample_plan(x_source, y_target)
    else:
        generation_node = make_node(generation_model)
        with torch.no_grad():
            fwd_traj = generation_node.trajectory(x_source, t_span=solve_t_span)
        generated_target = fwd_traj[-1].detach()
        generated_target, y_target, fwd_traj_guess, _ = ot_sampler.sample_plan_with_labels(
            generated_target,
            y_target,
            y0=fwd_traj.permute(1, 0, 2).contiguous(),
            y1=None,
        )
        x_source = fwd_traj_guess[:, 0, :].detach()
        fwd_traj_guess = fwd_traj_guess.permute(1, 0, 2).contiguous()
        mu_guess, mu_dot_guess = trajectory_velocity_guesses(
            generation_model,
            fwd_traj_guess,
            solve_t_span,
            y_target,
            x_source,
            reverse=True,
        )

    path, x0_keep, x1_keep, states = solve_paths(
        y_target,
        x_source,
        label=f'epoch {epoch} batch {batch_idx} bwd',
        mu_guess=mu_guess,
        mu_dot_guess=mu_dot_guess,
    )
    return {
        'path': path,
        'x0': x0_keep,
        'x1': x1_keep,
        'states': states,
        'label': f'epoch {epoch} batch {batch_idx} bwd',
    }


def train_fwd_on_path_cache(cache):
    return train_on_cached_path_pairs(
        fwd_model,
        fwd_optimizer,
        cache['path'],
        cache['x0'],
        cache['x1'],
        n_iters,
        cache['label'],
        ema=fwd_ema,
        batch_size=batch_size,
        device=device,
        log_every=200,
        no_pairs_message="no successful BVP pairs to train on",
    )


def train_bwd_on_path_cache(cache):
    return train_on_cached_path_pairs(
        bwd_model,
        bwd_optimizer,
        cache['path'],
        cache['x0'],
        cache['x1'],
        n_iters,
        cache['label'],
        ema=bwd_ema,
        batch_size=batch_size,
        device=device,
        log_every=200,
        no_pairs_message="no successful BVP pairs to train on",
    )


for epoch in range(n_epochs):
    print(f'\n=== Alternating epoch {epoch + 1} / {n_epochs} ===')
    train_fwd = (epoch % 2 == 0)

    source_data = source_dist.sample((n_dataset,))
    target_data = target_dist.sample((n_dataset,))
    source_loader = DataLoader(TotalDataset(source_data), batch_size=batch_size, shuffle=True, drop_last=True)
    target_loader = DataLoader(TotalDataset(target_data), batch_size=batch_size, shuffle=True, drop_last=True)

    if train_fwd:
        bwd_generation_model, generation_model = select_generation_model(
            bwd_ema, bwd_model, 'bwd_ema.ema_model', 'bwd_model'
        )
        print_ema_diagnostic(f'epoch {epoch} fwd cache generation', generation_model=generation_model)
        for i, (y_target, x_source) in enumerate(zip(target_loader, source_loader)):
            fwd_cache = generate_fwd_path_cache(epoch, i, y_target, x_source, bwd_generation_model)
            last_fwd_path = fwd_cache['path']
            last_fwd_x0, last_fwd_x1 = fwd_cache['x0'], fwd_cache['x1']
            last_fwd_states = fwd_cache['states']
            fwd_losses.extend(train_fwd_on_path_cache(fwd_cache))

            with torch.no_grad():
                fwd_model.eval()
                fwd_node = make_node(fwd_model)
                fwd_traj = fwd_node.trajectory(x0_eval, t_span=eval_t_span)
            plot_learned_three_body_trajectories(
                fwd_traj, x0_eval, 'Forward source to target', n_show=fwd_traj.shape[1]
            )
    else:
        fwd_generation_model, generation_model = select_generation_model(
            fwd_ema, fwd_model, 'fwd_ema.ema_model', 'fwd_model'
        )
        print_ema_diagnostic(f'epoch {epoch} bwd cache generation', generation_model=generation_model)
        for i, (x_source, y_target) in enumerate(zip(source_loader, target_loader)):
            bwd_cache = generate_bwd_path_cache(epoch, i, x_source, y_target, fwd_generation_model)
            last_bwd_path = bwd_cache['path']
            last_bwd_x0, last_bwd_x1 = bwd_cache['x0'], bwd_cache['x1']
            last_bwd_states = bwd_cache['states']
            bwd_losses.extend(train_bwd_on_path_cache(bwd_cache))

            with torch.no_grad():
                bwd_model.eval()
                bwd_node = make_node(bwd_model)
                bwd_traj = bwd_node.trajectory(x1_eval, t_span=eval_t_span)
            plot_learned_three_body_trajectories(
                bwd_traj, x1_eval, 'Backward target to source', n_show=bwd_traj.shape[1]
            )


In [20]:
# last_fwd_path = None
# last_bwd_path = None
# last_fwd_states = None
# last_bwd_states = None
# last_fwd_x0, last_fwd_x1 = None, None
# last_bwd_x0, last_bwd_x1 = None, None

# for epoch in range(n_epochs):
#     print(f'\n=== Alternating epoch {epoch + 1} / {n_epochs} ===')

#     if epoch % 2 == 0:
#         # Train fwd using couplings induced by the current bwd NODE.
#         y_target = sample_target(n_dataset)
#         bwd_model.eval()
#         bwd_node = make_node(bwd_model)
#         with torch.no_grad():
#             bwd_traj = bwd_node.trajectory(y_target, t_span=solve_t_span)
#         generated_source = bwd_traj[-1].detach()

#         last_fwd_path, last_fwd_x0, last_fwd_x1, last_fwd_states = solve_paths(
#             generated_source,
#             y_target,
#             label=f'epoch {epoch} fwd',
#         )
#         fwd_losses.extend(
#             train_on_cached_paths(
#                 fwd_model,
#                 fwd_optimizer,
#                 last_fwd_path,
#                 last_fwd_x0,
#                 last_fwd_x1,
#                 n_iters,
#                 label=f'epoch {epoch} fwd',
#             )
#         )
#     else:
#         # Train bwd using couplings induced by the current fwd NODE.
#         x_source = sample_source(n_dataset)
#         fwd_model.eval()
#         fwd_node = make_node(fwd_model)
#         with torch.no_grad():
#             fwd_traj = fwd_node.trajectory(x_source, t_span=solve_t_span)
#         generated_target = fwd_traj[-1].detach()

#         last_bwd_path, last_bwd_x0, last_bwd_x1, last_bwd_states = solve_paths(
#             generated_target,
#             x_source,
#             label=f'epoch {epoch} bwd',
#         )
#         bwd_losses.extend(
#             train_on_cached_paths(
#                 bwd_model,
#                 bwd_optimizer,
#                 last_bwd_path,
#                 last_bwd_x0,
#                 last_bwd_x1,
#                 n_iters,
#                 label=f'epoch {epoch} bwd',
#             )
#         )

## Training Losses

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.semilogy(fwd_losses, label='fwd')
plt.semilogy(bwd_losses, label='bwd')
plt.xlabel('model update')
plt.ylabel('flow matching loss')
plt.title('Training losses')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## Learned Flow Evaluation

In [131]:
# fwd_ema.ema_model.eval()
# bwd_ema.ema_model.eval()
# fwd_node = make_node(fwd_ema.ema_model)
# bwd_node = make_node(bwd_ema.ema_model)
fwd_model.eval()
bwd_model.eval()
fwd_node = make_node(fwd_model)
bwd_node = make_node(bwd_model)
eval_t_span = torch.linspace(0, 1, 100, device=device)

x0_eval = sample_source(eval_batch)
x0_eval2 = sample_source(eval_batch)
x1_eval = sample_target(eval_batch)
x1_eval2 = sample_target(eval_batch)

with torch.no_grad():
    fwd_traj = fwd_node.trajectory(x0_eval, t_span=eval_t_span)
    bwd_traj = bwd_node.trajectory(x1_eval, t_span=eval_t_span)

# fwd_traj = to_numpy(fwd_traj)
# bwd_traj = to_numpy(bwd_traj)

In [ ]:
w2_empirical_nu = wasserstein(x1_eval2,x1_eval,method='exact')
w2_empirical_mu = wasserstein(x0_eval2,x0_eval,method='exact')

w2_fwd = wasserstein(fwd_traj[-1], x1_eval, method="exact")
w2_bwd = wasserstein(bwd_traj[-1], x0_eval, method="exact")

print("W2 fwd endpoint-target:", w2_fwd)
print("W2 target empirical:", w2_empirical_nu)
print("relative excess fwd:", (w2_fwd - w2_empirical_nu) / (w2_empirical_nu + 1e-6))

print("W2 bwd endpoint-source:", w2_bwd)
print("W2 source empirical:", w2_empirical_mu)
print("relative excess bwd:", (w2_bwd - w2_empirical_mu) / (w2_empirical_mu + 1e-6))

print("fwd W2 / empirical:", w2_fwd / (w2_empirical_nu + 1e-6))
print("bwd W2 / empirical:", w2_bwd / (w2_empirical_mu + 1e-6))

In [ ]:
plot_learned_three_body_trajectories(fwd_traj, x1_eval, 'Forward source to target',n_show = fwd_traj.shape[1])
plot_learned_three_body_trajectories(bwd_traj, x0_eval, 'Backward target to source',n_show = bwd_traj.shape[1])

## Forward NODE vs Direct Hamiltonian BVP

In [ ]:
comparison_batch = min(eval_batch//10, fwd_traj.shape[1])
learned_traj = fwd_traj[:, :comparison_batch]
x0_cmp = x0_eval[:comparison_batch]
generated_target = learned_traj[-1].detach()

cmp_path = make_path()
cmp_path.batch_solve(x0_cmp, generated_target)
keep = cmp_path.success_mask.to(device=device)
x0_cmp_keep = x0_cmp[keep]
generated_target_keep = generated_target[keep]
learned_traj_keep = learned_traj[:, keep]
if x0_cmp_keep.shape[0] == 0:
    raise RuntimeError('No successful comparison BVPs.')
closed_form_traj = cached_mean_trajectory(cmp_path, x0_cmp_keep, generated_target_keep, eval_t_span)

per_time_rmse = (learned_traj_keep - closed_form_traj).pow(2).mean(dim=(1, 2)).sqrt()
trajectory_rmse = per_time_rmse.pow(2).mean().sqrt().item()
print(f'Forward NODE vs direct BVP trajectory RMSE: {trajectory_rmse:.6e}')
print(f'Comparison BVPs kept: {x0_cmp_keep.shape[0]} / {comparison_batch}')

n_compare_lines = min(100, x0_cmp_keep.shape[0])
learned_particles = to_numpy(as_particles(learned_traj_keep[:, :n_compare_lines]))
closed_particles = to_numpy(as_particles(closed_form_traj[:, :n_compare_lines]))
time_rmse = to_numpy(per_time_rmse)
t_eval_plot = to_numpy(eval_t_span)
colors = particle_colors()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for i in range(n_compare_lines):
    for pidx, color in enumerate(colors):
        axes[0].plot(learned_particles[:, i, pidx, 0], learned_particles[:, i, pidx, 1], color=color, alpha=0.16, linewidth=0.8)
        axes[1].plot(closed_particles[:, i, pidx, 0], closed_particles[:, i, pidx, 1], color=color, alpha=0.16, linewidth=0.8)

for pidx, color in enumerate(colors):
    axes[0].scatter(learned_particles[0, :, pidx, 0], learned_particles[0, :, pidx, 1], s=7, color=color, alpha=0.55)
    axes[0].scatter(learned_particles[-1, :, pidx, 0], learned_particles[-1, :, pidx, 1], s=9, color='black', marker='x', alpha=0.65)
    axes[1].scatter(closed_particles[0, :, pidx, 0], closed_particles[0, :, pidx, 1], s=7, color=color, alpha=0.55)
    axes[1].scatter(closed_particles[-1, :, pidx, 0], closed_particles[-1, :, pidx, 1], s=9, color='black', marker='x', alpha=0.65)

axes[0].set_title('Learned forward NODE trajectories')
axes[1].set_title('Direct fixed-center BVP trajectories')
axes[2].plot(t_eval_plot, time_rmse, color='black', linewidth=2.0)
axes[2].set_xlabel('t')
axes[2].set_ylabel('RMSE')
axes[2].set_title('Per-time trajectory RMSE')
axes[2].grid(alpha=0.25)

for ax in axes[:2]:
    ax.scatter([fixed_position[0].item()], [fixed_position[1].item()], marker='*', s=160, color='gold', edgecolor='black')
    ax.axis('equal')
    ax.grid(alpha=0.25)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.tight_layout()
plt.show()

## Hamiltonian Diagnostics

In [ ]:
def trajectory_hamiltonian_per_sample(traj, t_span, potential):
    dt = t_span[1] - t_span[0]
    velocity = torch.empty_like(traj)
    velocity[0] = (traj[1] - traj[0]) / dt
    velocity[-1] = (traj[-1] - traj[-2]) / dt
    velocity[1:-1] = (traj[2:] - traj[:-2]) / (2.0 * dt)
    kinetic = 0.5 * velocity.pow(2).sum(dim=-1)
    potential_energy = torch.stack([potential.energy(x_t) for x_t in traj], dim=0)
    return kinetic + potential_energy


diagnostic_count = min(diagnostic_batch, fwd_traj.shape[1])
H = trajectory_hamiltonian_per_sample(fwd_traj[:, :diagnostic_count], eval_t_span, potential)
H_np = to_numpy(H)
n_hamiltonian_traces = min(30, diagnostic_count, H_np.shape[1])

plt.figure(figsize=(8, 4))
plt.plot(to_numpy(eval_t_span), H_np[:, :n_hamiltonian_traces], color="0.25", alpha=0.22, linewidth=0.8)
plt.plot(to_numpy(eval_t_span), H_np.mean(axis=1), color="black", linewidth=2.0, label="mean")
plt.xlabel("t")
plt.ylabel("H(t)")
plt.title("Approximate Hamiltonian along learned forward trajectories")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()